# Imports


In [9]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
# Should print T4, not empty

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [10]:
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [11]:
DATA_FOLDER = "/content/drive/MyDrive/EV_PROJECT/Ready_for_LSTM"

client_names = ["client_A", "client_B1", "client_B2", "client_B3"]

#Loading all client data

In [12]:
import gc

client_names = ["client_A", "client_B1", "client_B2", "client_B3"]

print("Verifying client data shapes:\n")
for name in client_names:
    folder = os.path.join(DATA_FOLDER, name)
    X = np.load(os.path.join(folder, "X_train.npy"), mmap_mode='r')
    y = np.load(os.path.join(folder, "y_train.npy"), mmap_mode='r')
    print(f"  {name} — X_train: {X.shape} | y_train: {y.shape}")
    del X, y

gc.collect()
print("\nShapes verified. Data not held in RAM.")

Verifying client data shapes:

  client_A — X_train: (356682, 60, 20) | y_train: (356682,)
  client_B1 — X_train: (220744, 60, 20) | y_train: (220744,)
  client_B2 — X_train: (157288, 60, 20) | y_train: (157288,)
  client_B3 — X_train: (86042, 60, 20) | y_train: (86042,)

Shapes verified. Data not held in RAM.


#Build the LSTM model
We define this as a function so each client
can get a fresh copy with the same architecture

In [13]:
# Get input shape from one client without loading full array
sample_X = np.load(os.path.join(DATA_FOLDER, "client_A", "X_train.npy"), mmap_mode='r')
input_shape = sample_X.shape[1:]   # (60, 20)
del sample_X
gc.collect()

print("Input shape:", input_shape)

def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

global_model = build_lstm_model(input_shape)
global_model.summary()

Input shape: (60, 20)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 60, 128)        │        76,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 60, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,657 (541.63 KB)

 Trainable params: 138,657 (541.63 KB)

 Non-trainable params: 0 (0.00 B)

#FedAvg helper
Averages client weights weighted by their number of training samples

In [14]:
def federated_average(client_weights, client_sizes):
    """
    client_weights : list of weight arrays from each client
    client_sizes   : list of training sample counts per client
    Returns averaged weights (same structure as model.get_weights())
    """
    total_samples = sum(client_sizes)
    avg_weights = []

    for layer_idx in range(len(client_weights[0])):
        layer_avg = sum(
            client_weights[i][layer_idx] * (client_sizes[i] / total_samples)
            for i in range(len(client_weights))
        )
        avg_weights.append(layer_avg)

    return avg_weights

# Federated Learning training loop

In [ ]:
FL_ROUNDS    = 20
LOCAL_EPOCHS = 5
BATCH_SIZE   = 32

round_mae_history = []

os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs(f"{DATA_FOLDER}/checkpoints", exist_ok=True)

local_model = build_lstm_model(input_shape)

# One reusable local model — don't create a new one every round
local_model = build_lstm_model(input_shape)

for round_num in range(1, FL_ROUNDS + 1):

    print(f"\n========== FL Round {round_num}/{FL_ROUNDS} ==========")

    client_weight_list = []
    client_size_list   = []

    for name in client_names:
        folder = os.path.join(DATA_FOLDER, name)

        # Load only this client's data — float32 to halve RAM usage
        X_train = np.load(os.path.join(folder, "X_train.npy")).astype(np.float32)
        y_train = np.load(os.path.join(folder, "y_train.npy")).astype(np.float32)

        # Give local model current global weights
        local_model.set_weights(global_model.get_weights())

        # Train locally
        local_model.fit(
            X_train, y_train,
            epochs     = LOCAL_EPOCHS,
            batch_size = BATCH_SIZE,
            verbose    = 0
        )

        client_weight_list.append(local_model.get_weights())
        client_size_list.append(len(X_train))
        print(f"  {name} done — {len(X_train)} samples")

        # Free immediately — critical
        del X_train, y_train
        gc.collect()

    # Aggregate
    global_model.set_weights(federated_average(client_weight_list, client_size_list))
    del client_weight_list
    gc.collect()

    # Evaluate on all test sets — one client at a time
    all_mae = []
    for name in client_names:
        folder = os.path.join(DATA_FOLDER, name)
        X_test = np.load(os.path.join(folder, "X_test.npy")).astype(np.float32)
        y_test = np.load(os.path.join(folder, "y_test.npy")).astype(np.float32)
        _, mae = global_model.evaluate(X_test, y_test, verbose=0)
        all_mae.append(mae)
        del X_test, y_test
        gc.collect()

    avg_mae = np.mean(all_mae)
    round_mae_history.append(avg_mae)
    print(f"  Avg Test MAE across all clients: {avg_mae:.4f}")

    global_model.save_weights(f"/content/checkpoints/global_round_{round_num}.weights.h5")
    shutil.copy(
        f"/content/checkpoints/global_round_{round_num}.weights.h5",
        f"{DATA_FOLDER}/checkpoints/global_round_{round_num}.weights.h5"
    )
    print(f"  Checkpoint saved to Drive — round {round_num}")

del local_model
gc.collect()
print("\nFederated Learning complete.")


========== FL Round 1/20 ==========
  client_A done — 356682 samples
  client_B1 done — 220744 samples


#Per-client final evaluation

In [ ]:
print("Final Global Model — Per-Client Results:\n")
for name in client_names:
    folder = os.path.join(DATA_FOLDER, name)
    X_test = np.load(os.path.join(folder, "X_test.npy")).astype(np.float32)
    y_test = np.load(os.path.join(folder, "y_test.npy")).astype(np.float32)
    loss, mae = global_model.evaluate(X_test, y_test, verbose=0)
    print(f"  {name}  →  MAE: {mae:.4f}  |  MSE: {loss:.4f}")
    del X_test, y_test
    gc.collect()

#Plots

In [ ]:
# Plot 1: MAE across rounds
plt.figure(figsize=(9, 4))
plt.plot(range(1, FL_ROUNDS + 1), round_mae_history, marker='o', color='steelblue')
plt.title("Global Model MAE Across Federated Rounds")
plt.xlabel("FL Round")
plt.ylabel("Test MAE (scaled RDR)")
plt.grid(True)
plt.tight_layout()
plt.savefig("fl_round_mae.png", dpi=150)
plt.show()

# Plot 2: Predicted vs Actual on client_A
folder = os.path.join(DATA_FOLDER, "client_A")
X_test = np.load(os.path.join(folder, "X_test.npy")).astype(np.float32)
y_test = np.load(os.path.join(folder, "y_test.npy")).astype(np.float32)

y_pred = global_model.predict(X_test).flatten()

plt.figure(figsize=(10, 4))
plt.plot(y_test[:500],  label="Actual RDR",    alpha=0.8)
plt.plot(y_pred[:500],  label="Predicted RDR", alpha=0.8)
plt.title("Predicted vs Actual RDR — client_A test set (first 500 samples)")
plt.xlabel("Timestep")
plt.ylabel("RDR (scaled)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("rdr_prediction_clientA.png", dpi=150)
plt.show()

del X_test, y_test
gc.collect()